# LoanLens🔍 EDA & ML Project

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

# Client Loan Application Dataset

## Dataset Overview

In [2]:
df_train = pd.read_csv('application_train.csv')
print(df_train.shape)
print(df_train.columns)

(307511, 122)
Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='str', length=122)


In [3]:
df_test = pd.read_csv('application_test.csv')
print(df_test.shape)
print(df_test.columns)

(48744, 121)
Index(['SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
       'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
       'AMT_ANNUITY', 'AMT_GOODS_PRICE',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='str', length=121)


In [4]:
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

print('In train only:', train_cols - test_cols)
print('In test only: ', test_cols - train_cols)
print('Common:       ', len(train_cols & test_cols))

In train only: {'TARGET'}
In test only:  set()
Common:        121


## 📋 Data Dictionary — `application_train.csv` & `application_test.csv`

These two tables describe the **clients** (one row per client). `application_train` has **122 columns** (307,511 clients) and `application_test` has the same columns **without `TARGET`** (121 columns, 48,744 clients). We combine them into one clients table. The columns are grouped by topic below: loan info, personal info, income & work, assets & housing, region, contact flags, external scores, building info, social circle, documents and credit-bureau history.

| # | Column | Description | Data Type | Notes |
|---|--------|-------------|-----------|-------|
| 1 | `SK_ID_CURR` | Unique ID of the client | Numerical (ID) | 🔑 **Key to join with `previous_application`** |
| 2 | `TARGET` | 1 = client had payment difficulties (late > X days on at least one installment), 0 = all other cases | Categorical (binary) | **Train only**, not our target → dropped |
| 3 | `NAME_CONTRACT_TYPE` | Type of the client's current loan: *Cash loans* or *Revolving loans* | Categorical | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 4 | `AMT_CREDIT` | Credit amount of the client's current loan | Numerical (continuous) | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 5 | `AMT_ANNUITY` | Annuity (installment) of the client's current loan | Numerical (continuous) | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 6 | `AMT_GOODS_PRICE` | Price of the goods for the client's current loan | Numerical (continuous) | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 7 | `CODE_GENDER` | Gender of the client (M / F) | Categorical | Contains a few rare `XNA` values |
| 8 | `DAYS_BIRTH` | Client's age in days at the time of application (negative = days before application) | Numerical | Negative values → convert to `AGE` in years |
| 9 | `NAME_FAMILY_STATUS` | Family status (Married, Single, Civil marriage, Separated, Widow) | Categorical | Contains `Unknown` |
| 10 | `CNT_CHILDREN` | Number of children the client has | Numerical (discrete) |  |
| 11 | `CNT_FAM_MEMBERS` | Number of family members of the client | Numerical (discrete) |  |
| 12 | `NAME_EDUCATION_TYPE` | Highest level of education achieved | Categorical (ordinal) |  |
| 13 | `NAME_TYPE_SUITE` | Who accompanied the client when applying (Unaccompanied, Family, Spouse, ...) | Categorical | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 14 | `AMT_INCOME_TOTAL` | Total annual income of the client | Numerical (continuous) | Has extreme outliers |
| 15 | `NAME_INCOME_TYPE` | Income type (Working, Commercial associate, Pensioner, State servant, ...) | Categorical |  |
| 16 | `OCCUPATION_TYPE` | Client's occupation (Laborers, Sales staff, Managers, Drivers, ...) | Categorical | ~31% missing |
| 17 | `ORGANIZATION_TYPE` | Type of organization where the client works | Categorical | ~58 categories, contains `XNA` (mostly pensioners) |
| 18 | `DAYS_EMPLOYED` | How many days before the application the client started the current job | Numerical | Anomaly: `365243` (≈1000 years) for unemployed/pensioners |
| 19 | `FLAG_OWN_CAR` | Does the client own a car? (Y / N) | Categorical (binary) |  |
| 20 | `OWN_CAR_AGE` | Age of the client's car in years | Numerical | Missing when client has no car |
| 21 | `FLAG_OWN_REALTY` | Does the client own a house or flat? (Y / N) | Categorical (binary) |  |
| 22 | `NAME_HOUSING_TYPE` | Housing situation (House/apartment, Rented, With parents, Municipal, ...) | Categorical |  |
| 23 | `REGION_POPULATION_RELATIVE` | Normalized population of the region where the client lives (higher = more populated) | Numerical |  |
| 24 | `REGION_RATING_CLIENT` | Home Credit's rating of the region where the client lives (1, 2, 3) | Numerical (ordinal) |  |
| 25 | `REGION_RATING_CLIENT_W_CITY` | Home Credit's rating of the region, taking the city into account (1, 2, 3) | Numerical (ordinal) |  |
| 26 | `REG_REGION_NOT_LIVE_REGION` | 1 if permanent address ≠ contact address (region level) | Categorical (binary flag) |  |
| 27 | `REG_REGION_NOT_WORK_REGION` | 1 if permanent address ≠ work address (region level) | Categorical (binary flag) |  |
| 28 | `LIVE_REGION_NOT_WORK_REGION` | 1 if contact address ≠ work address (region level) | Categorical (binary flag) |  |
| 29 | `REG_CITY_NOT_LIVE_CITY` | 1 if permanent address ≠ contact address (city level) | Categorical (binary flag) |  |
| 30 | `REG_CITY_NOT_WORK_CITY` | 1 if permanent address ≠ work address (city level) | Categorical (binary flag) |  |
| 31 | `LIVE_CITY_NOT_WORK_CITY` | 1 if contact address ≠ work address (city level) | Categorical (binary flag) |  |
| 32 | `DAYS_REGISTRATION` | How many days before the application the client changed their registration | Numerical | Negative values |
| 33 | `DAYS_ID_PUBLISH` | How many days before the application the client changed the ID document | Numerical | Negative values |
| 34 | `DAYS_LAST_PHONE_CHANGE` | How many days before the application the client changed phone | Numerical | Negative values |
| 35 | `FLAG_MOBIL` | Did the client provide a mobile phone? (1 / 0) | Categorical (binary flag) | Almost constant (≈ all 1) |
| 36 | `FLAG_EMP_PHONE` | Did the client provide an employer (work) phone? (1 / 0) | Categorical (binary flag) |  |
| 37 | `FLAG_WORK_PHONE` | Did the client provide a work phone? (1 / 0) | Categorical (binary flag) |  |
| 38 | `FLAG_CONT_MOBILE` | Was the mobile phone reachable? (1 / 0) | Categorical (binary flag) | Almost constant |
| 39 | `FLAG_PHONE` | Did the client provide a home phone? (1 / 0) | Categorical (binary flag) |  |
| 40 | `FLAG_EMAIL` | Did the client provide an email? (1 / 0) | Categorical (binary flag) |  |
| 41 | `WEEKDAY_APPR_PROCESS_START` | Day of the week the client applied for the loan | Categorical | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 42 | `HOUR_APPR_PROCESS_START` | Approximate hour the client applied for the loan | Numerical (discrete) | ⚠️ Same name in `previous_application` (here it describes the client's current loan) |
| 43 | `EXT_SOURCE_1` | Normalized credit score from an external data source #1 (0 – 1) | Numerical | ~56% missing |
| 44 | `EXT_SOURCE_2` | Normalized credit score from an external data source #2 (0 – 1) | Numerical |  |
| 45 | `EXT_SOURCE_3` | Normalized credit score from an external data source #3 (0 – 1) | Numerical | ~20% missing |
| 46 | `APARTMENTS_AVG` | Building where the client lives: apartment size (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 47 | `BASEMENTAREA_AVG` | Building where the client lives: basement area (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 48 | `YEARS_BEGINEXPLUATATION_AVG` | Building where the client lives: years since the building started being used (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 49 | `YEARS_BUILD_AVG` | Building where the client lives: age of the building (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 50 | `COMMONAREA_AVG` | Building where the client lives: common area (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 51 | `ELEVATORS_AVG` | Building where the client lives: number of elevators (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 52 | `ENTRANCES_AVG` | Building where the client lives: number of entrances (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 53 | `FLOORSMAX_AVG` | Building where the client lives: maximum number of floors (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 54 | `FLOORSMIN_AVG` | Building where the client lives: minimum number of floors (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 55 | `LANDAREA_AVG` | Building where the client lives: land area (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 56 | `LIVINGAPARTMENTS_AVG` | Building where the client lives: living apartments (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 57 | `LIVINGAREA_AVG` | Building where the client lives: living area (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 58 | `NONLIVINGAPARTMENTS_AVG` | Building where the client lives: non-living apartments (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 59 | `NONLIVINGAREA_AVG` | Building where the client lives: non-living area (average, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 60 | `APARTMENTS_MODE` | Building where the client lives: apartment size (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 61 | `BASEMENTAREA_MODE` | Building where the client lives: basement area (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 62 | `YEARS_BEGINEXPLUATATION_MODE` | Building where the client lives: years since the building started being used (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 63 | `YEARS_BUILD_MODE` | Building where the client lives: age of the building (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 64 | `COMMONAREA_MODE` | Building where the client lives: common area (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 65 | `ELEVATORS_MODE` | Building where the client lives: number of elevators (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 66 | `ENTRANCES_MODE` | Building where the client lives: number of entrances (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 67 | `FLOORSMAX_MODE` | Building where the client lives: maximum number of floors (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 68 | `FLOORSMIN_MODE` | Building where the client lives: minimum number of floors (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 69 | `LANDAREA_MODE` | Building where the client lives: land area (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 70 | `LIVINGAPARTMENTS_MODE` | Building where the client lives: living apartments (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 71 | `LIVINGAREA_MODE` | Building where the client lives: living area (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 72 | `NONLIVINGAPARTMENTS_MODE` | Building where the client lives: non-living apartments (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 73 | `NONLIVINGAREA_MODE` | Building where the client lives: non-living area (mode, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 74 | `APARTMENTS_MEDI` | Building where the client lives: apartment size (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 75 | `BASEMENTAREA_MEDI` | Building where the client lives: basement area (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 76 | `YEARS_BEGINEXPLUATATION_MEDI` | Building where the client lives: years since the building started being used (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 77 | `YEARS_BUILD_MEDI` | Building where the client lives: age of the building (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 78 | `COMMONAREA_MEDI` | Building where the client lives: common area (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 79 | `ELEVATORS_MEDI` | Building where the client lives: number of elevators (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 80 | `ENTRANCES_MEDI` | Building where the client lives: number of entrances (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 81 | `FLOORSMAX_MEDI` | Building where the client lives: maximum number of floors (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 82 | `FLOORSMIN_MEDI` | Building where the client lives: minimum number of floors (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 83 | `LANDAREA_MEDI` | Building where the client lives: land area (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 84 | `LIVINGAPARTMENTS_MEDI` | Building where the client lives: living apartments (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 85 | `LIVINGAREA_MEDI` | Building where the client lives: living area (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 86 | `NONLIVINGAPARTMENTS_MEDI` | Building where the client lives: non-living apartments (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 87 | `NONLIVINGAREA_MEDI` | Building where the client lives: non-living area (median, normalized) | Numerical | Building info: high % missing (≈50–70%) |
| 88 | `FONDKAPREMONT_MODE` | Building where the client lives: capital-repair fund type (normalized) | Categorical | Building info: high % missing |
| 89 | `HOUSETYPE_MODE` | Building where the client lives: house type (block of flats, specific housing, ...) | Categorical | Building info: high % missing |
| 90 | `TOTALAREA_MODE` | Building where the client lives: total area (normalized) | Numerical | Building info: high % missing |
| 91 | `WALLSMATERIAL_MODE` | Building where the client lives: wall material (Panel, Stone/brick, ...) | Categorical | Building info: high % missing |
| 92 | `EMERGENCYSTATE_MODE` | Building where the client lives: is it in emergency state? (Yes / No) | Categorical | Building info: high % missing |
| 93 | `OBS_30_CNT_SOCIAL_CIRCLE` | Number of people in the client's social surroundings observed with 30 DPD (days past due) default | Numerical (discrete) |  |
| 94 | `DEF_30_CNT_SOCIAL_CIRCLE` | Number of people in the client's social surroundings who defaulted on 30 DPD | Numerical (discrete) |  |
| 95 | `OBS_60_CNT_SOCIAL_CIRCLE` | Number of people in the client's social surroundings observed with 60 DPD default | Numerical (discrete) |  |
| 96 | `DEF_60_CNT_SOCIAL_CIRCLE` | Number of people in the client's social surroundings who defaulted on 60 DPD | Numerical (discrete) |  |
| 97 | `FLAG_DOCUMENT_2` | Did the client provide document #2? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 98 | `FLAG_DOCUMENT_3` | Did the client provide document #3? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 99 | `FLAG_DOCUMENT_4` | Did the client provide document #4? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 100 | `FLAG_DOCUMENT_5` | Did the client provide document #5? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 101 | `FLAG_DOCUMENT_6` | Did the client provide document #6? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 102 | `FLAG_DOCUMENT_7` | Did the client provide document #7? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 103 | `FLAG_DOCUMENT_8` | Did the client provide document #8? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 104 | `FLAG_DOCUMENT_9` | Did the client provide document #9? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 105 | `FLAG_DOCUMENT_10` | Did the client provide document #10? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 106 | `FLAG_DOCUMENT_11` | Did the client provide document #11? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 107 | `FLAG_DOCUMENT_12` | Did the client provide document #12? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 108 | `FLAG_DOCUMENT_13` | Did the client provide document #13? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 109 | `FLAG_DOCUMENT_14` | Did the client provide document #14? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 110 | `FLAG_DOCUMENT_15` | Did the client provide document #15? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 111 | `FLAG_DOCUMENT_16` | Did the client provide document #16? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 112 | `FLAG_DOCUMENT_17` | Did the client provide document #17? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 113 | `FLAG_DOCUMENT_18` | Did the client provide document #18? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 114 | `FLAG_DOCUMENT_19` | Did the client provide document #19? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 115 | `FLAG_DOCUMENT_20` | Did the client provide document #20? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 116 | `FLAG_DOCUMENT_21` | Did the client provide document #21? (1 / 0) | Categorical (binary flag) | Mostly 0, document meaning not disclosed |
| 117 | `AMT_REQ_CREDIT_BUREAU_HOUR` | Number of enquiries to the Credit Bureau about the client in the 1 hour before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |
| 118 | `AMT_REQ_CREDIT_BUREAU_DAY` | Number of enquiries to the Credit Bureau about the client in the 1 day (excluding the last hour) before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |
| 119 | `AMT_REQ_CREDIT_BUREAU_WEEK` | Number of enquiries to the Credit Bureau about the client in the 1 week (excluding the last day) before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |
| 120 | `AMT_REQ_CREDIT_BUREAU_MON` | Number of enquiries to the Credit Bureau about the client in the 1 month (excluding the last week) before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |
| 121 | `AMT_REQ_CREDIT_BUREAU_QRT` | Number of enquiries to the Credit Bureau about the client in the 3 months (excluding the last month) before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |
| 122 | `AMT_REQ_CREDIT_BUREAU_YEAR` | Number of enquiries to the Credit Bureau about the client in the 1 year (excluding the last 3 months) before application | Numerical (discrete) | Credit report info, ~13% missing (in train) |

> **Note:** All `DAYS_*` columns are recorded **relative to the application date**, so they are negative (e.g. `DAYS_BIRTH = -12000` means the client was born ≈ 33 years before applying).

In [5]:
df_pre_application = pd.read_csv('previous_application.csv')
print(df_pre_application.shape)
print(df_pre_application.columns)

(1670214, 37)
Index(['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY',
       'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY',
       'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY',
       'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE',
       'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE',
       'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE',
       'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE',
       'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY',
       'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION',
       'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
       'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL'],
      dtype='str')


In [6]:
prev_cols = set(df_pre_application.columns)

print('Common with train:', train_cols & prev_cols)

Common with train: {'NAME_CONTRACT_TYPE', 'SK_ID_CURR', 'NAME_TYPE_SUITE', 'AMT_CREDIT', 'AMT_ANNUITY', 'WEEKDAY_APPR_PROCESS_START', 'AMT_GOODS_PRICE', 'HOUR_APPR_PROCESS_START'}


## 📋 Data Dictionary — `previous_application.csv`

Each row is **one previous loan application** of a client (a client can have several). It contains **37 columns** and **1,670,214 applications**. All `DAYS_*` columns are relative to the client's current application (negative = days before).

| # | Column | Description | Data Type | Notes |
|---|--------|-------------|-----------|-------|
| 1 | `SK_ID_PREV` | Unique ID of the previous application | Numerical (ID) | Identifier only |
| 2 | `SK_ID_CURR` | ID of the client (same as in `application_train/test`) | Numerical (ID) | 🔑 Key: join with the clients table |
| 3 | `NAME_CONTRACT_TYPE` | Product type: Cash loans, Consumer loans (POS), Revolving loans | Categorical | ✅ Input (known at application time) |
| 4 | `AMT_ANNUITY` | Annuity (installment) of the previous application | Numerical | ⚠️ Check for leakage: depends on the credit amount and term |
| 5 | `AMT_APPLICATION` | **How much credit the client asked for** | Numerical | ✅ Input (known at application time): the requested amount, key input for the idea |
| 6 | `AMT_CREDIT` | Final credit amount on the previous application (the client may ask for X and the bank approves Y) | Numerical | ❌ **Leakage** (known only after the decision): decided by the bank |
| 7 | `AMT_DOWN_PAYMENT` | Down payment on the previous application | Numerical | ✅ Input (known at application time), high % missing |
| 8 | `AMT_GOODS_PRICE` | Price of the goods the client asked for (if applicable) | Numerical | ✅ Input (known at application time) |
| 9 | `WEEKDAY_APPR_PROCESS_START` | Day of the week the client applied | Categorical |  |
| 10 | `HOUR_APPR_PROCESS_START` | Approximate hour the client applied | Numerical |  |
| 11 | `FLAG_LAST_APPL_PER_CONTRACT` | Was it the last application for the contract? (sometimes a contract has several applications by mistake) | Categorical (Y/N) |  |
| 12 | `NFLAG_LAST_APPL_IN_DAY` | Was it the client's last application that day? (1/0) | Categorical (0/1) |  |
| 13 | `RATE_DOWN_PAYMENT` | Down payment rate (normalized) | Numerical | High % missing |
| 14 | `RATE_INTEREST_PRIMARY` | Interest rate (normalized) | Numerical | ❌ **Leakage** (known only after the decision), ~99% missing |
| 15 | `RATE_INTEREST_PRIVILEGED` | Privileged interest rate (normalized) | Numerical | ❌ **Leakage** (known only after the decision), ~99% missing |
| 16 | `NAME_CASH_LOAN_PURPOSE` | Purpose of the cash loan | Categorical | Mostly `XAP` / `XNA` |
| 17 | `NAME_CONTRACT_STATUS` | Status of the application: Approved, Refused, Canceled, Unused offer | Categorical | 🎯 **Target**: keep Approved / Refused |
| 18 | `DAYS_DECISION` | When the decision was made, in days before the current application | Numerical | Use only to order applications in time (not as input) |
| 19 | `NAME_PAYMENT_TYPE` | Payment method the client chose for the previous application | Categorical |  |
| 20 | `CODE_REJECT_REASON` | Why the application was rejected | Categorical | ❌ **Leakage** (known only after the decision): filled only for refused |
| 21 | `NAME_TYPE_SUITE` | Who accompanied the client when applying | Categorical |  |
| 22 | `NAME_CLIENT_TYPE` | Was the client new or old when applying (New, Repeater, Refreshed) | Categorical | ✅ Input (known at application time) |
| 23 | `NAME_GOODS_CATEGORY` | Kind of goods the client applied for | Categorical |  |
| 24 | `NAME_PORTFOLIO` | Portfolio of the application: Cash, POS, Cards, Cars | Categorical | ⚠️ Check for leakage |
| 25 | `NAME_PRODUCT_TYPE` | Was the application x-sell or walk-in | Categorical |  |
| 26 | `CHANNEL_TYPE` | Channel through which the client was acquired | Categorical |  |
| 27 | `SELLERPLACE_AREA` | Selling area of the seller | Numerical |  |
| 28 | `NAME_SELLER_INDUSTRY` | Industry of the seller | Categorical |  |
| 29 | `CNT_PAYMENT` | Term of the previous credit (number of payments) | Numerical | ⚠️ Check for leakage |
| 30 | `NAME_YIELD_GROUP` | Interest rate group: low, middle, high | Categorical | ⚠️ Check for leakage: set by the bank |
| 31 | `PRODUCT_COMBINATION` | Detailed product combination | Categorical |  |
| 32 | `DAYS_FIRST_DRAWING` | When the first disbursement was made | Numerical | ❌ **Leakage** (known only after the decision): only for approved loans |
| 33 | `DAYS_FIRST_DUE` | When the first payment was due | Numerical | ❌ **Leakage** (known only after the decision) |
| 34 | `DAYS_LAST_DUE_1ST_VERSION` | When the first planned last payment was due | Numerical | ❌ **Leakage** (known only after the decision) |
| 35 | `DAYS_LAST_DUE` | When the last payment was due | Numerical | ❌ **Leakage** (known only after the decision) |
| 36 | `DAYS_TERMINATION` | Expected termination of the credit | Numerical | ❌ **Leakage** (known only after the decision) |
| 37 | `NFLAG_INSURED_ON_APPROVAL` | Did the client request insurance on approval (1/0) | Categorical (0/1) | ❌ **Leakage** (known only after the decision) |

> **Leakage rule:** for the approve / refuse decision, we only use information the bank has **when the client applies**. Anything decided or recorded **after** the decision (final amount, rejection reason, payment dates, interest rate) cannot be an input.

In [7]:
# Combine train and test into one clients table.
df_train.drop('TARGET', axis=1, inplace=True)

df_clients = pd.concat([df_train, df_test], ignore_index=True)
df_clients.shape

(356255, 121)

In [8]:
df_clients['SK_ID_CURR'].duplicated().sum()

np.int64(0)

- **Kept only Approved and Refused:** Canceled (the client cancelled before a decision) and Unused offer (the client did not use the approved offer) do not represent the bank's decision.

In [9]:
# Filter the previous application data to only include rows where the contract status is either 'Approved' or 'Refused'.
df_pre_application = df_pre_application[df_pre_application['NAME_CONTRACT_STATUS'].isin(['Approved', 'Refused'])].copy()

print(df_pre_application.shape)
print(df_pre_application['NAME_CONTRACT_STATUS'].value_counts())

(1327459, 37)
NAME_CONTRACT_STATUS
Approved    1036781
Refused      290678
Name: count, dtype: int64


In [10]:
check_num = ['AMT_ANNUITY', 'CNT_PAYMENT', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE']

df_pre_application.groupby('NAME_CONTRACT_STATUS')[check_num].apply(lambda g: g.isna().mean() * 100).round(1)

,AMT_ANNUITY,CNT_PAYMENT,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE
NAME_CONTRACT_STATUS,,,,,
Approved,0.0,0.0,0.0,36.1,4.1
Refused,14.1,14.1,0.0,69.9,12.8


In [11]:
check_cat = ['NAME_YIELD_GROUP', 'NAME_PORTFOLIO', 'NAME_CASH_LOAN_PURPOSE', 'NAME_GOODS_CATEGORY']

for col in check_cat:
    print(col)
    print(pd.crosstab(df_pre_application[col] == 'XNA', df_pre_application['NAME_CONTRACT_STATUS'], normalize='columns').round(3) * 100)
    print('-' * 50)

NAME_YIELD_GROUP
NAME_CONTRACT_STATUS  Approved  Refused
NAME_YIELD_GROUP                       
False                     90.6     69.9
True                       9.4     30.1
--------------------------------------------------
NAME_PORTFOLIO
NAME_CONTRACT_STATUS  Approved  Refused
NAME_PORTFOLIO                         
False                    100.0     85.9
True                       0.0     14.1
--------------------------------------------------
NAME_CASH_LOAN_PURPOSE
NAME_CONTRACT_STATUS    Approved  Refused
NAME_CASH_LOAN_PURPOSE                   
False                       72.5     57.0
True                        27.5     43.0
--------------------------------------------------
NAME_GOODS_CATEGORY
NAME_CONTRACT_STATUS  Approved  Refused
NAME_GOODS_CATEGORY                    
False                     60.4     23.0
True                      39.6     77.0
--------------------------------------------------


- **Incomplete applications:** 40,901 applications had no annuity, no term and `NAME_PORTFOLIO = XNA`, and 99.99% of them were refused (early rejection). Dropped, because an incomplete application will never reach the model in the app, and keeping them would let the model learn "missing annuity → refused" (leakage).

In [12]:
# Drop incomplete applications (no annuity, no term, no portfolio): 40,901 rows, 99.99% refused early
cond = (df_pre_application['AMT_ANNUITY'].isna()) & \
       (df_pre_application['CNT_PAYMENT'].isna()) & \
       (df_pre_application['NAME_PORTFOLIO'] == 'XNA')

df_pre_application = df_pre_application[~cond]

print(df_pre_application.shape)
print(df_pre_application['NAME_CONTRACT_STATUS'].value_counts())

(1286558, 37)
NAME_CONTRACT_STATUS
Approved    1036777
Refused      249781
Name: count, dtype: int64


In [13]:
# Check: is the client's current loan inside previous_application? (same client, credit and annuity)
check = df_clients[['SK_ID_CURR', 'AMT_CREDIT', 'AMT_ANNUITY']].merge(
            df_pre_application[['SK_ID_CURR', 'AMT_CREDIT', 'AMT_ANNUITY']],
            on=['SK_ID_CURR', 'AMT_CREDIT', 'AMT_ANNUITY'], how='inner')

print(check['SK_ID_CURR'].nunique())

591


- Only ~591 clients (0.17%) have a previous application with the same credit and annuity as their current loan, so `previous_application` holds the clients' **earlier** applications, not the current loan. The current-loan columns (`AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `NAME_CONTRACT_TYPE`) are **future information** for the previous applications, so they were **not selected**.

In [14]:
personal_cols = ['CODE_GENDER', 'DAYS_BIRTH', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE']
work_cols     = ['AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'DAYS_EMPLOYED']
assets_cols   = ['FLAG_OWN_CAR', 'OWN_CAR_AGE', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE']
region_cols   = ['REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT_W_CITY', 'LIVE_REGION_NOT_WORK_REGION']
credit_cols   = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
                 'AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']
doc_cols      = [f'FLAG_DOCUMENT_{i}' for i in range(2, 22)]
contact_cols  = ['FLAG_EMP_PHONE']

client_cols = personal_cols + work_cols + assets_cols + region_cols + credit_cols + doc_cols + contact_cols

df_clients = df_clients.set_index('SK_ID_CURR')[client_cols]
df_clients.shape

(356255, 48)

In [15]:
prev_cols_sel = ['SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION',
                 'AMT_APPLICATION', 'CNT_PAYMENT', 'AMT_GOODS_PRICE', 'AMT_DOWN_PAYMENT',
                 'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE',
                 'AMT_ANNUITY', 'AMT_CREDIT']

df_pre_application = df_pre_application[prev_cols_sel]
df_pre_application.shape

(1286558, 11)

In [16]:
# Latest application per client (DAYS_DECISION closest to 0)
idx_last = df_pre_application.groupby('SK_ID_CURR')['DAYS_DECISION'].idxmax()

df_last = df_pre_application.loc[idx_last].set_index('SK_ID_CURR')

print(df_last.shape)
print(df_last['NAME_CONTRACT_STATUS'].value_counts(normalize=True) * 100)

(338347, 10)
NAME_CONTRACT_STATUS
Approved    84.748202
Refused     15.251798
Name: proportion, dtype: float64


In [17]:
df = df_clients.join(df_last, how='left')

print(df.shape)
print(df['NAME_CONTRACT_STATUS'].value_counts(dropna=False))

(356255, 58)
NAME_CONTRACT_STATUS
Approved    286743
Refused      51604
NaN          17908
Name: count, dtype: int64


In [18]:
df.to_parquet('Clients_LoanLens.parquet')

In [19]:
df.head()

,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100002,M,-9461,Single / not married,0,1.0,Secondary / secondary special,202500.0,Working,Laborers,Business Entity Type 3,-637,N,NaN,Y,House / apartment,0.018801,2,0,0.083037,0.262949,0.139376,0.0,0.0,0.0,0.0,0.0,1.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Approved,-606.0,179055.0,24.0,179055.0,0.0,Consumer loans,New,9251.775,179055.0
100003,F,-16765,Married,0,2.0,Higher education,270000.0,State servant,Core staff,School,-1188,N,NaN,N,House / apartment,0.003541,1,0,0.311267,0.622246,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Approved,-746.0,900000.0,12.0,900000.0,NaN,Cash loans,Repeater,98356.995,1035882.0
100004,M,-19046,Single / not married,0,1.0,Secondary / secondary special,67500.0,Working,Laborers,Government,-225,Y,26.0,Y,House / apartment,0.010032,2,0,NaN,0.555912,0.729567,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Approved,-815.0,24282.0,4.0,24282.0,4860.0,Consumer loans,New,5357.250,20106.0
100006,F,-19005,Civil marriage,0,2.0,Secondary / secondary special,135000.0,Working,Laborers,Business Entity Type 3,-3039,N,NaN,Y,House / apartment,0.008019,2,0,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Approved,-181.0,675000.0,48.0,675000.0,NaN,Cash loans,Repeater,24246.000,675000.0
100007,M,-19932,Single / not married,0,1.0,Secondary / secondary special,121500.0,Working,Core staff,Religion,-3038,N,NaN,Y,House / apartment,0.028663,2,0,NaN,0.322738,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Approved,-374.0,247500.0,24.0,247500.0,NaN,Cash loans,Repeater,16037.640,274288.5


# Data Exploration 

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 356255 entries, 100002 to 456250
Data columns (total 58 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   CODE_GENDER                  356255 non-null  str    
 1   DAYS_BIRTH                   356255 non-null  int64  
 2   NAME_FAMILY_STATUS           356255 non-null  str    
 3   CNT_CHILDREN                 356255 non-null  int64  
 4   CNT_FAM_MEMBERS              356253 non-null  float64
 5   NAME_EDUCATION_TYPE          356255 non-null  str    
 6   AMT_INCOME_TOTAL             356255 non-null  float64
 7   NAME_INCOME_TYPE             356255 non-null  str    
 8   OCCUPATION_TYPE              244259 non-null  str    
 9   ORGANIZATION_TYPE            356255 non-null  str    
 10  DAYS_EMPLOYED                356255 non-null  int64  
 11  FLAG_OWN_CAR                 356255 non-null  str    
 12  OWN_CAR_AGE                  121014 non-null  float64
 13  FLAG_OWN_R

- Check Duplicates 

In [21]:
df.duplicated().sum()

np.int64(0)

- Check Missing Values Percentage

In [22]:
df.isna().mean().round(4)* 100

CODE_GENDER                     0.00
DAYS_BIRTH                      0.00
NAME_FAMILY_STATUS              0.00
CNT_CHILDREN                    0.00
CNT_FAM_MEMBERS                 0.00
NAME_EDUCATION_TYPE             0.00
AMT_INCOME_TOTAL                0.00
NAME_INCOME_TYPE                0.00
OCCUPATION_TYPE                31.44
ORGANIZATION_TYPE               0.00
DAYS_EMPLOYED                   0.00
FLAG_OWN_CAR                    0.00
OWN_CAR_AGE                    66.03
FLAG_OWN_REALTY                 0.00
NAME_HOUSING_TYPE               0.00
REGION_POPULATION_RELATIVE      0.00
REGION_RATING_CLIENT_W_CITY     0.00
LIVE_REGION_NOT_WORK_REGION     0.00
EXT_SOURCE_1                   54.43
EXT_SOURCE_2                    0.19
EXT_SOURCE_3                   19.55
AMT_REQ_CREDIT_BUREAU_HOUR     13.35
AMT_REQ_CREDIT_BUREAU_DAY      13.35
AMT_REQ_CREDIT_BUREAU_WEEK     13.35
AMT_REQ_CREDIT_BUREAU_MON      13.35
AMT_REQ_CREDIT_BUREAU_QRT      13.35
AMT_REQ_CREDIT_BUREAU_YEAR     13.35
F

- Explore Numeric Columns

In [23]:
df.describe().round(2)

,DAYS_BIRTH,CNT_CHILDREN,CNT_FAM_MEMBERS,AMT_INCOME_TOTAL,DAYS_EMPLOYED,OWN_CAR_AGE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,FLAG_EMP_PHONE,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,AMT_ANNUITY,AMT_CREDIT
count,356255.00,356255.00,356253.00,3.562550e+05,356255.00,121014.00,356255.00,356255.00,356255.00,162345.00,355587.00,286622.00,308687.00,308687.00,308687.00,308687.00,308687.00,308687.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.0,356255.00,356255.0,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,356255.00,338347.00,338347.00,338347.00,329981.00,177108.00,338344.00,338347.00
mean,-16041.25,0.41,2.15,1.701161e+05,64317.23,12.02,0.02,2.03,0.04,0.50,0.51,0.51,0.01,0.01,0.03,0.23,0.30,1.91,0.00,0.72,0.00,0.02,0.09,0.00,0.08,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.01,0.00,0.01,0.00,0.00,0.00,0.82,-563.33,259631.65,17.22,266231.13,7054.85,17352.11,287465.94
std,4358.80,0.72,0.91,2.235068e+05,141705.53,11.88,0.01,0.50,0.20,0.21,0.19,0.19,0.08,0.10,0.19,0.86,0.79,1.87,0.01,0.45,0.01,0.12,0.28,0.01,0.27,0.06,0.0,0.06,0.0,0.06,0.05,0.03,0.09,0.02,0.08,0.02,0.02,0.02,0.39,553.25,333895.75,15.69,335508.41,21620.96,15322.24,364046.94
min,-25229.00,0.00,1.00,2.565000e+04,-17912.00,0.00,0.00,-1.00,0.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-2922.00,0.00,0.00,0.00,-0.45,0.00,0.00
25%,-19676.00,0.00,2.00,1.125000e+05,-2781.00,5.00,0.01,2.00,0.00,0.34,0.39,0.37,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,-692.00,61066.35,6.00,65605.50,0.00,7026.45,65718.00
50%,-15755.00,0.00,2.00,1.530000e+05,-1224.00,9.00,0.02,2.00,0.00,0.51,0.56,0.53,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,-387.00,131760.00,12.00,135000.00,0.00,12495.92,139239.00
75%,-12425.00,1.00,3.00,2.025000e+05,-290.00,15.00,0.03,2.00,0.00,0.67,0.66,0.67,0.00,0.00,0.00,0.00,0.00,3.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,-214.00,307125.00,24.00,315000.00,8010.93,22549.66,349096.50
max,-7338.00,20.00,21.00,1.170000e+08,365243.00,91.00,0.07,3.00,1.00,0.96,0.85,0.90,4.00,9.00,8.00,27.00,261.00,25.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,-1.00,4050000.00,84.00,4050000.00,2475000.00,300425.44,4104351.00


- Explore Categorical Columns

In [24]:
df.describe(include= 'str')

,CODE_GENDER,NAME_FAMILY_STATUS,NAME_EDUCATION_TYPE,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,NAME_CONTRACT_STATUS,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE
count,356255,356255,356255,356255,244259,356255,356255,356255,356255,338347,338347,338347
unique,3,6,5,8,18,58,2,2,6,2,3,4
top,F,Married,Secondary / secondary special,Working,Laborers,Business Entity Type 3,N,Y,House / apartment,Approved,Consumer loans,Repeater
freq,235126,228715,252379,183307,63841,78832,235235,246970,316513,286743,174124,228625


In [25]:
df[doc_cols].sum(axis=1).value_counts()

1    318053
0     30296
2      7742
3       163
4         1
Name: count, dtype: int64

In [26]:
(df[doc_cols].mean() * 100).sort_values(ascending=False).round(2)

FLAG_DOCUMENT_3     72.05
FLAG_DOCUMENT_6      8.80
FLAG_DOCUMENT_8      8.23
FLAG_DOCUMENT_5      1.51
FLAG_DOCUMENT_16     0.86
FLAG_DOCUMENT_18     0.72
FLAG_DOCUMENT_9      0.40
FLAG_DOCUMENT_11     0.35
FLAG_DOCUMENT_13     0.30
FLAG_DOCUMENT_14     0.25
FLAG_DOCUMENT_15     0.10
FLAG_DOCUMENT_19     0.05
FLAG_DOCUMENT_20     0.04
FLAG_DOCUMENT_21     0.03
FLAG_DOCUMENT_17     0.02
FLAG_DOCUMENT_7      0.02
FLAG_DOCUMENT_4      0.01
FLAG_DOCUMENT_2      0.00
FLAG_DOCUMENT_10     0.00
FLAG_DOCUMENT_12     0.00
dtype: float64

In [27]:
# Keep the 4 most common documents, and count the rest in one column
main_docs  = ['FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_5']
other_docs = [c for c in doc_cols if c not in main_docs]   # 16 rare documents

df['OTHER_DOCUMENTS'] = df[other_docs].sum(axis=1)
df.drop(columns=other_docs, inplace=True)

print(df.shape)
print(df['OTHER_DOCUMENTS'].value_counts())

(356255, 43)
OTHER_DOCUMENTS
0    345176
1     10857
2       221
3         1
Name: count, dtype: int64


- **Documents:** 89% of clients provided exactly one document, mostly `FLAG_DOCUMENT_3` (72%), `6` (8.8%), `8` (8.2%) or `5` (1.5%). Kept these 4, and summed the 16 rare documents (each < 1%) into `OTHER_DOCUMENTS` → 20 columns become 5.

In [28]:
bureau_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
               'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
               'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']

print(df[bureau_cols].isnull().all(axis=1).sum())
print(df[bureau_cols].isnull().any(axis=1).sum())

47568
47568


In [29]:
# 1) Flag: has a credit bureau record? (before filling, so the info is not lost)
df['HAS_BUREAU_RECORD'] = df[bureau_cols].notnull().any(axis=1).astype(int)

# 2) No record = no enquiries
df[bureau_cols] = df[bureau_cols].fillna(0)

# 3) Total enquiries in the last year (periods don't overlap)
df['TOTAL_BUREAU_REQ'] = df[bureau_cols].sum(axis=1).astype(int)

df.drop(columns=bureau_cols, inplace=True)

print(df.shape)
print(df['HAS_BUREAU_RECORD'].value_counts())

(356255, 39)
HAS_BUREAU_RECORD
1    308687
0     47568
Name: count, dtype: int64


- **Credit Bureau:** the 6 `AMT_REQ_CREDIT_BUREAU_*` columns are missing for the same clients (no bureau record). Created `HAS_BUREAU_RECORD` (1/0), filled missing with 0, and summed the 6 periods (they don't overlap) into `TOTAL_BUREAU_REQ` → 6 columns become 2.

In [30]:
df.duplicated().sum()

np.int64(0)

In [31]:
df.describe().round(2)

,DAYS_BIRTH,CNT_CHILDREN,CNT_FAM_MEMBERS,AMT_INCOME_TOTAL,DAYS_EMPLOYED,OWN_CAR_AGE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ
count,356255.00,356255.00,356253.00,3.562550e+05,356255.00,121014.00,356255.00,356255.00,356255.00,162345.00,355587.00,286622.00,356255.00,356255.00,356255.00,356255.00,356255.00,338347.00,338347.00,338347.00,329981.00,177108.00,338344.00,338347.00,356255.00,356255.00,356255.00
mean,-16041.25,0.41,2.15,1.701161e+05,64317.23,12.02,0.02,2.03,0.04,0.50,0.51,0.51,0.72,0.02,0.09,0.08,0.82,-563.33,259631.65,17.22,266231.13,7054.85,17352.11,287465.94,0.03,0.87,2.16
std,4358.80,0.72,0.91,2.235068e+05,141705.53,11.88,0.01,0.50,0.20,0.21,0.19,0.19,0.45,0.12,0.28,0.27,0.39,553.25,333895.75,15.69,335508.41,21620.96,15322.24,364046.94,0.18,0.34,2.26
min,-25229.00,0.00,1.00,2.565000e+04,-17912.00,0.00,0.00,-1.00,0.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-2922.00,0.00,0.00,0.00,-0.45,0.00,0.00,0.00,0.00,0.00
25%,-19676.00,0.00,2.00,1.125000e+05,-2781.00,5.00,0.01,2.00,0.00,0.34,0.39,0.37,0.00,0.00,0.00,0.00,1.00,-692.00,61066.35,6.00,65605.50,0.00,7026.45,65718.00,0.00,1.00,0.00
50%,-15755.00,0.00,2.00,1.530000e+05,-1224.00,9.00,0.02,2.00,0.00,0.51,0.56,0.53,1.00,0.00,0.00,0.00,1.00,-387.00,131760.00,12.00,135000.00,0.00,12495.92,139239.00,0.00,1.00,2.00
75%,-12425.00,1.00,3.00,2.025000e+05,-290.00,15.00,0.03,2.00,0.00,0.67,0.66,0.67,1.00,0.00,0.00,0.00,1.00,-214.00,307125.00,24.00,315000.00,8010.93,22549.66,349096.50,0.00,1.00,3.00
max,-7338.00,20.00,21.00,1.170000e+08,365243.00,91.00,0.07,3.00,1.00,0.96,0.85,0.90,1.00,1.00,1.00,1.00,1.00,-1.00,4050000.00,84.00,4050000.00,2475000.00,300425.44,4104351.00,3.00,1.00,262.00


- Indepth Check for Numerical Columns 

In [32]:
num_cols = df.select_dtypes(include= 'number').columns
num_cols

Index(['DAYS_BIRTH', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL',
       'DAYS_EMPLOYED', 'OWN_CAR_AGE', 'REGION_POPULATION_RELATIVE',
       'REGION_RATING_CLIENT_W_CITY', 'LIVE_REGION_NOT_WORK_REGION',
       'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'FLAG_DOCUMENT_3',
       'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8',
       'FLAG_EMP_PHONE', 'DAYS_DECISION', 'AMT_APPLICATION', 'CNT_PAYMENT',
       'AMT_GOODS_PRICE', 'AMT_DOWN_PAYMENT', 'AMT_ANNUITY', 'AMT_CREDIT',
       'OTHER_DOCUMENTS', 'HAS_BUREAU_RECORD', 'TOTAL_BUREAU_REQ'],
      dtype='str')

- **Plots on a sample:** Plotly saves every data point inside the notebook. With 356K rows, the histogram + box loop made the notebook grow to ~99 MB (in a previous version), close to GitHub's 100 MB file limit and very slow to open. So the plots use a random sample of 10,000 clients, which shows the same distribution shape. A sample can miss very rare extreme values, so the real minimum and maximum are taken from `describe()` on all the data above.

In [33]:
# Random sample of 10K clients: same distribution shape, light notebook
df_check = df.sample(10000, random_state= 42)

for col in num_cols:
    px.histogram(data_frame= df_check, x= col, title= col).show()
    px.box(data_frame= df_check, x= col, title= col).show()

- Indepth Check for Categorical Columns 

In [34]:
cat_cols = df.select_dtypes(include= 'str').columns
cat_cols

Index(['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE',
       'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE',
       'NAME_CONTRACT_STATUS', 'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE'],
      dtype='str')

In [35]:
for col in cat_cols: 
    print(col) 
    print(df[col].nunique()) 
    print(df[col].unique()) 
    print('-' * 50)

CODE_GENDER
3
<ArrowStringArray>
['M', 'F', 'XNA']
Length: 3, dtype: str
--------------------------------------------------
NAME_FAMILY_STATUS
6
<ArrowStringArray>
['Single / not married',              'Married',       'Civil marriage',
                'Widow',            'Separated',              'Unknown']
Length: 6, dtype: str
--------------------------------------------------
NAME_EDUCATION_TYPE
5
<ArrowStringArray>
['Secondary / secondary special',              'Higher education',
             'Incomplete higher',               'Lower secondary',
               'Academic degree']
Length: 5, dtype: str
--------------------------------------------------
NAME_INCOME_TYPE
8
<ArrowStringArray>
[             'Working',        'State servant', 'Commercial associate',
            'Pensioner',           'Unemployed',              'Student',
          'Businessman',      'Maternity leave']
Length: 8, dtype: str
--------------------------------------------------
OCCUPATION_TYPE
18
<ArrowStri

In [36]:
df.head()

,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100002,M,-9461,Single / not married,0,1.0,Secondary / secondary special,202500.0,Working,Laborers,Business Entity Type 3,-637,N,NaN,Y,House / apartment,0.018801,2,0,0.083037,0.262949,0.139376,1,0,0,0,1,Approved,-606.0,179055.0,24.0,179055.0,0.0,Consumer loans,New,9251.775,179055.0,0,1,1
100003,F,-16765,Married,0,2.0,Higher education,270000.0,State servant,Core staff,School,-1188,N,NaN,N,House / apartment,0.003541,1,0,0.311267,0.622246,NaN,1,0,0,0,1,Approved,-746.0,900000.0,12.0,900000.0,NaN,Cash loans,Repeater,98356.995,1035882.0,0,1,0
100004,M,-19046,Single / not married,0,1.0,Secondary / secondary special,67500.0,Working,Laborers,Government,-225,Y,26.0,Y,House / apartment,0.010032,2,0,NaN,0.555912,0.729567,0,0,0,0,1,Approved,-815.0,24282.0,4.0,24282.0,4860.0,Consumer loans,New,5357.250,20106.0,0,1,0
100006,F,-19005,Civil marriage,0,2.0,Secondary / secondary special,135000.0,Working,Laborers,Business Entity Type 3,-3039,N,NaN,Y,House / apartment,0.008019,2,0,NaN,0.650442,NaN,1,0,0,0,1,Approved,-181.0,675000.0,48.0,675000.0,NaN,Cash loans,Repeater,24246.000,675000.0,0,0,0
100007,M,-19932,Single / not married,0,1.0,Secondary / secondary special,121500.0,Working,Core staff,Religion,-3038,N,NaN,Y,House / apartment,0.028663,2,0,NaN,0.322738,NaN,0,0,0,1,1,Approved,-374.0,247500.0,24.0,247500.0,NaN,Cash loans,Repeater,16037.640,274288.5,0,1,0


## Data Cleaning

In [37]:
print(df['REGION_RATING_CLIENT_W_CITY'].value_counts())

REGION_RATING_CLIENT_W_CITY
 2    265260
 3     50652
 1     40342
-1         1
Name: count, dtype: int64


In [38]:
df[df['REGION_RATING_CLIENT_W_CITY'] == -1]

,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
224393,F,-21636,Single / not married,0,1.0,Secondary / secondary special,270000.0,Commercial associate,Managers,Business Entity Type 2,-5630,N,NaN,N,Municipal apartment,0.000253,-1,0,0.522436,0.46432,0.544235,1,0,0,0,1,Approved,-670.0,1035000.0,30.0,1035000.0,NaN,Cash loans,Repeater,53084.205,1110141.0,0,1,0


In [39]:
# Drop 1 client with an invalid region rating (-1); valid values are 1, 2, 3
df = df[df['REGION_RATING_CLIENT_W_CITY'] != -1]

print(df.shape)
print(df['REGION_RATING_CLIENT_W_CITY'].value_counts())

(356254, 39)
REGION_RATING_CLIENT_W_CITY
2    265260
3     50652
1     40342
Name: count, dtype: int64


In [40]:
cond_emp = df['DAYS_EMPLOYED'] == 365243

print(cond_emp.sum())
print(df.loc[cond_emp, 'NAME_INCOME_TYPE'].value_counts())
print(df.loc[cond_emp, 'ORGANIZATION_TYPE'].value_counts())

64648
NAME_INCOME_TYPE
Pensioner     64625
Unemployed       23
Name: count, dtype: int64
ORGANIZATION_TYPE
XNA    64648
Name: count, dtype: int64


In [41]:
# 365243 = not currently working (pensioners & unemployed) → 0 months employed
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, 0)
df['MONTH_EMPLOYED'] = df['DAYS_EMPLOYED'].abs() // 30

# Same clients have ORGANIZATION_TYPE = 'XNA' → new category
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].replace('XNA', 'Not Working')

print(df['MONTH_EMPLOYED'].describe().round(2))
print((df['ORGANIZATION_TYPE'] == 'Not Working').sum())

count    356254.00
mean         65.00
std          76.77
min           0.00
25%           9.00
50%          40.00
75%          92.00
max         597.00
Name: MONTH_EMPLOYED, dtype: float64
64648


In [42]:
df['AGE_YEARS'] = (df['DAYS_BIRTH'].abs() // 365.25).astype(int)
df['AGE_YEARS'].describe().round(2)

count    356254.00
mean         43.42
std          11.93
min          20.00
25%          34.00
50%          43.00
75%          53.00
max          69.00
Name: AGE_YEARS, dtype: float64

- **Region rating:** dropped 1 client with an invalid rating (-1).
- **DAYS_EMPLOYED = 365243:** 64,648 clients (18%) not currently working (64,625 pensioners + 23 unemployed), the same clients as `ORGANIZATION_TYPE = XNA`. Replaced with 0, converted to `MONTH_EMPLOYED`, and `XNA` → `'Not Working'`.
- **Age:** `DAYS_BIRTH` (negative days) → `AGE_YEARS` (20–69).

In [43]:
df['CODE_GENDER'].unique()

<ArrowStringArray>
['M', 'F', 'XNA']
Length: 3, dtype: str

In [44]:
df['CODE_GENDER'].value_counts()

CODE_GENDER
F      235125
M      121125
XNA         4
Name: count, dtype: int64

In [45]:
# Drop clients with unknown gender (XNA)
df = df[df['CODE_GENDER'] != 'XNA']

print(df.shape)
print(df['CODE_GENDER'].value_counts())

(356250, 41)
CODE_GENDER
F    235125
M    121125
Name: count, dtype: int64


In [46]:
df['CNT_FAM_MEMBERS'].describe().round()

count    356248.0
mean          2.0
std           1.0
min           1.0
25%           2.0
50%           2.0
75%           3.0
max          21.0
Name: CNT_FAM_MEMBERS, dtype: float64

In [47]:
(df['NAME_FAMILY_STATUS'] == 'Unknown').value_counts()

NAME_FAMILY_STATUS
False    356248
True          2
Name: count, dtype: int64

In [48]:
df[df['CNT_FAM_MEMBERS'].isna()]

,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,DAYS_DECISION,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ,MONTH_EMPLOYED,AGE_YEARS
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
148605,M,-12396,Unknown,0,NaN,Lower secondary,450000.0,Commercial associate,Managers,Insurance,-1161,N,NaN,Y,Municipal apartment,0.015221,2,1,0.628564,0.700618,NaN,0,1,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,38,33
317181,F,-12844,Unknown,0,NaN,Higher education,202500.0,Commercial associate,Accountants,Business Entity Type 2,-232,N,NaN,Y,House / apartment,0.031329,2,0,NaN,0.645168,0.670652,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,7,35


In [49]:
(df['NAME_FAMILY_STATUS'] == 'Unknown').sum()

np.int64(2)

In [50]:
# Drop 2 clients with no family info at all (CNT_FAM_MEMBERS = NaN and NAME_FAMILY_STATUS = 'Unknown')
df = df.dropna(subset=['CNT_FAM_MEMBERS'])

In [51]:
print(df.shape)                                          
print(df['CNT_FAM_MEMBERS'].isnull().sum())             
print((df['NAME_FAMILY_STATUS'] == 'Unknown').sum())   

(356248, 41)
0
0


In [52]:
df['CNT_FAM_MEMBERS'] = df['CNT_FAM_MEMBERS'].astype(int)

- **Gender:** dropped 4 clients with `XNA`.
- **Family:** dropped 2 clients with `NAME_FAMILY_STATUS = Unknown` and `CNT_FAM_MEMBERS = NaN` (same clients, no family info at all).

In [53]:
print(df['NAME_CLIENT_TYPE'].value_counts(dropna=False))
print(df.loc[df['NAME_CLIENT_TYPE'] == 'XNA', 'NAME_CONTRACT_STATUS'].value_counts())

NAME_CLIENT_TYPE
Repeater     228621
New           71931
Refreshed     37222
NaN           17906
XNA             568
Name: count, dtype: int64
NAME_CONTRACT_STATUS
Approved    444
Refused     124
Name: count, dtype: int64


In [54]:
# NAME_CLIENT_TYPE = 'XNA' (568 rows, 0.16%) → NaN, will be imputed with the mode inside the pipeline
df['NAME_CLIENT_TYPE'] = df['NAME_CLIENT_TYPE'].replace('XNA', np.nan)

print(df['NAME_CLIENT_TYPE'].value_counts(dropna=False))

NAME_CLIENT_TYPE
Repeater     228621
New           71931
Refreshed     37222
NaN           18474
Name: count, dtype: int64


In [55]:
df['ORGANIZATION_TYPE'].unique()

<ArrowStringArray>
['Business Entity Type 3',                 'School',             'Government',
               'Religion',                  'Other',            'Not Working',
            'Electricity',               'Medicine', 'Business Entity Type 2',
          'Self-employed',      'Transport: type 2',           'Construction',
                'Housing',           'Kindergarten',          'Trade: type 7',
      'Industry: type 11',               'Military',               'Services',
    'Security Ministries',      'Transport: type 4',       'Industry: type 1',
              'Emergency',               'Security',          'Trade: type 2',
             'University',      'Transport: type 3',                 'Police',
 'Business Entity Type 1',                 'Postal',       'Industry: type 4',
            'Agriculture',             'Restaurant',                'Culture',
                  'Hotel',       'Industry: type 7',          'Trade: type 3',
       'Industry: type 3',       

In [56]:
# Remove meaningless numbered sub-types (e.g. 'Industry: type 11' → 'Industry')
df['ORGANIZATION_TYPE'] = (df['ORGANIZATION_TYPE']
                           .str.replace(r':?\s*[Tt]ype\s*\d+', '', regex=True)
                           .str.strip())

print(df['ORGANIZATION_TYPE'].nunique())
print(df['ORGANIZATION_TYPE'].unique())

35
<ArrowStringArray>
[    'Business Entity',              'School',          'Government',
            'Religion',               'Other',         'Not Working',
         'Electricity',            'Medicine',       'Self-employed',
           'Transport',        'Construction',             'Housing',
        'Kindergarten',               'Trade',            'Industry',
            'Military',            'Services', 'Security Ministries',
           'Emergency',            'Security',          'University',
              'Police',              'Postal',         'Agriculture',
          'Restaurant',             'Culture',               'Hotel',
                'Bank',           'Insurance',              'Mobile',
      'Legal Services',         'Advertising',            'Cleaning',
             'Telecom',             'Realtor']
Length: 35, dtype: str


- **ORGANIZATION_TYPE:** removed numbered sub-types (e.g. `Industry: type 11` → `Industry`): 58 → 35 categories.

In [57]:
(df['ORGANIZATION_TYPE'] == 'XNA').value_counts()

ORGANIZATION_TYPE
False    356248
Name: count, dtype: int64

In [58]:
print((df['ORGANIZATION_TYPE'] == 'XNA').sum())            
print((df['ORGANIZATION_TYPE'] == 'Not Working').sum())    
print(df['ORGANIZATION_TYPE'].nunique())  

0
64648
35


In [59]:
df['OCCUPATION_TYPE'].unique()

<ArrowStringArray>
[             'Laborers',            'Core staff',           'Accountants',
              'Managers',                     nan,               'Drivers',
           'Sales staff',        'Cleaning staff',         'Cooking staff',
 'Private service staff',        'Medicine staff',        'Security staff',
 'High skill tech staff',  'Waiters/barmen staff',    'Low-skill Laborers',
         'Realty agents',           'Secretaries',              'IT staff',
              'HR staff']
Length: 19, dtype: str

In [60]:
(df['OCCUPATION_TYPE'].isna()).value_counts()

OCCUPATION_TYPE
False    244254
True     111994
Name: count, dtype: int64

In [61]:
pd.crosstab(df['ORGANIZATION_TYPE'] == 'Not Working',
            df['OCCUPATION_TYPE'].isnull())

OCCUPATION_TYPE,False,True
ORGANIZATION_TYPE,,
False,244252,47348
True,2,64646


In [62]:
# Group A: not working + no occupation → 'Not Working'
cond_a = (df['ORGANIZATION_TYPE'] == 'Not Working') & (df['OCCUPATION_TYPE'].isnull())
df.loc[cond_a, 'OCCUPATION_TYPE'] = 'Not Working'

# Group B: not working but with an occupation (contradiction) → drop
cond_b = (df['ORGANIZATION_TYPE'] == 'Not Working') & (df['OCCUPATION_TYPE'] != 'Not Working')
df = df[~cond_b]

print(df.shape)
print((df['OCCUPATION_TYPE'] == 'Not Working').sum())
print(df['OCCUPATION_TYPE'].isnull().sum())

(356246, 41)
64646
47348


- **OCCUPATION_TYPE:** not working + no occupation (64,646) → `'Not Working'`; dropped 2 contradictory rows (not working but with an occupation); employed with no occupation (47,348, 13.3%) → `'Other'` in the pipeline.

In [63]:
pd.crosstab(df['FLAG_OWN_CAR'], df['OWN_CAR_AGE'].isnull())

OWN_CAR_AGE,False,True
FLAG_OWN_CAR,,
N,0,235228
Y,121012,6


In [64]:
# No car → car age = 0 (FLAG_OWN_CAR separates them from new cars)
df.loc[df['FLAG_OWN_CAR'] == 'N', 'OWN_CAR_AGE'] = 0

# Remaining NaN = car owners with unknown car age (6 rows) → drop
df = df.dropna(subset=['OWN_CAR_AGE'])

print(df.shape)
print(df['OWN_CAR_AGE'].isnull().sum())

(356240, 41)
0


- **OWN_CAR_AGE:** clients with no car (235,228) → 0 (`FLAG_OWN_CAR` separates them from new cars); dropped 6 car owners with unknown car age.

In [65]:
print(df['AMT_APPLICATION'].describe().round(2))
print((df['AMT_APPLICATION'] == 0).sum())
print(df.loc[df['AMT_APPLICATION'] == 0, 'NAME_CONTRACT_STATUS'].value_counts())
print(df.loc[df['AMT_APPLICATION'] == 0, 'NAME_CONTRACT_TYPE'].value_counts())

count     338334.00
mean      259628.28
std       333895.61
min            0.00
25%        61066.35
50%       131755.50
75%       307102.50
max      4050000.00
Name: AMT_APPLICATION, dtype: float64
9171
NAME_CONTRACT_STATUS
Approved    8169
Refused     1002
Name: count, dtype: int64
NAME_CONTRACT_TYPE
Revolving loans    9166
Cash loans            5
Name: count, dtype: int64


In [66]:
rev = df['NAME_CONTRACT_TYPE'] == 'Revolving loans'

print(rev.sum())
print((df.loc[rev, 'AMT_APPLICATION'] == 0).mean().round(4) * 100)

42945
21.34


In [67]:
# Drop applications with a requested amount of 0 (9,171 rows, 99.9% Revolving loans):
# no requested amount → the approve/refuse idea does not apply
df = df[df['AMT_APPLICATION'] != 0]

print(df.shape)
print((df['AMT_APPLICATION'] == 0).sum())

(347069, 41)
0


- **AMT_APPLICATION = 0:** 9,171 applications (99.9% Revolving loans, where the bank sets the limit and no amount is requested). Dropped, since the approve / refuse idea needs a requested amount.

In [68]:
has_app = df['NAME_CONTRACT_STATUS'].notnull()
prev_num = ['AMT_APPLICATION', 'CNT_PAYMENT', 'AMT_GOODS_PRICE', 'AMT_DOWN_PAYMENT', 'AMT_ANNUITY', 'AMT_CREDIT']

df[has_app].groupby('NAME_CONTRACT_STATUS')[prev_num].apply(lambda g: g.isna().mean() * 100).round(2)

,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,AMT_ANNUITY,AMT_CREDIT
NAME_CONTRACT_STATUS,,,,,,
Approved,0.0,0.0,0.0,39.78,0.0,0.0
Refused,0.0,0.0,0.0,81.53,0.0,0.0


In [69]:
df[has_app].groupby(['NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS'])['AMT_DOWN_PAYMENT'].apply(lambda s: s.isna().mean() * 100).round(1)

NAME_CONTRACT_TYPE  NAME_CONTRACT_STATUS
Cash loans          Approved                97.8
                    Refused                 98.7
Consumer loans      Approved                 0.0
                    Refused                  0.0
Revolving loans     Approved                98.6
                    Refused                 96.4
Name: AMT_DOWN_PAYMENT, dtype: float64

In [70]:
# Cash and Revolving loans have no down payment → 0 (only for clients who have an application)
cond_dp = df['NAME_CONTRACT_STATUS'].notnull() & df['AMT_DOWN_PAYMENT'].isnull()
df.loc[cond_dp, 'AMT_DOWN_PAYMENT'] = 0

print(df.loc[df['NAME_CONTRACT_STATUS'].notnull(), 'AMT_DOWN_PAYMENT'].isnull().sum())

0


- **AMT_DOWN_PAYMENT:** missing depends on the loan type, not the decision: Consumer loans always have a down payment (0% missing), Cash and Revolving loans almost never (~98% missing in both Approved and Refused). Filled with 0 (no down payment) for clients with an application.

In [71]:
df[has_app].groupby('NAME_CONTRACT_STATUS')['EXT_SOURCE_1'].apply(lambda s: s.isna().mean() * 100).round(2)

NAME_CONTRACT_STATUS
Approved    54.13
Refused     55.38
Name: EXT_SOURCE_1, dtype: float64

- **EXT_SOURCE_1:** 54% missing, but the missing rate is the same for Approved (54.1%) and Refused (55.4%), so missing values do not leak the decision.
→ Kept for the model notebook, where it will be tested with and without.

In [72]:
# DAYS_BIRTH / DAYS_EMPLOYED: replaced by AGE_YEARS / MONTH_EMPLOYED
# DAYS_DECISION: only used to find the latest application, not an app input
df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_DECISION'], inplace=True)

df.shape

(347069, 38)

- **Outliers** (e.g. `AMT_INCOME_TOTAL` max = 117M, `CNT_CHILDREN` max = 20) were **kept as instructed**, and will be handled in the model notebook (e.g. log transform).

In [73]:
df.to_parquet('Clients_LoanLens_clean.parquet')
print('Saved :', df.shape)

Saved : (347069, 38)


## 🧹 Cleaning Summary
- **Tables:** clients (train + test) + each client's **latest** application (Approved / Refused only, 40,901 incomplete applications dropped).
- **Rows:** 356,255 → **347,069** (dropped invalid / unknown values and 9,171 applications with a requested amount of 0).
- **Columns:** fixed `365243`, `XNA`, ages and missing values; documents 20 → 5; credit bureau 6 → 2; **outliers kept as instructed** (handled in the model notebook).
- **Final:** 347,069 × 38 → `Clients_LoanLens_clean.parquet`

## 📊 Exploratory Data Analysis (EDA)

In [74]:
# EDA copy: clients with an application only (Approved / Refused), df itself is not changed
df_eda = df[df['NAME_CONTRACT_STATUS'].notna()].copy()
df_eda['REFUSED'] = (df_eda['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

print(df_eda.shape)

(329163, 39)


In [75]:
# One sample for distribution plots (histogram / box / scatter)
df_sample = df_eda.sample(10000, random_state= 42)

### Univariate Analysis

In [76]:
# Distribution of 6 key columns (10K sample → light notebook)
for col in ['AGE_YEARS', 'AMT_INCOME_TOTAL', 'AMT_APPLICATION', 'EXT_SOURCE_3', 'CNT_PAYMENT', 'MONTH_EMPLOYED']:
    px.histogram(data_frame= df_sample, x= col, nbins= 50, title= f'Distribution of {col}',
                 log_y= col in ['AMT_INCOME_TOTAL', 'AMT_APPLICATION']).show()

In [77]:
df_eda[['AGE_YEARS', 'AMT_INCOME_TOTAL', 'AMT_APPLICATION', 'EXT_SOURCE_3', 'CNT_PAYMENT', 'MONTH_EMPLOYED']].describe().round(2)

,AGE_YEARS,AMT_INCOME_TOTAL,AMT_APPLICATION,EXT_SOURCE_3,CNT_PAYMENT,MONTH_EMPLOYED
count,329163.00,3.291630e+05,329163.00,265386.00,329163.00,329163.00
mean,43.46,1.686679e+05,266861.94,0.51,17.70,65.03
std,11.96,2.284740e+05,335651.72,0.19,15.64,76.59
min,20.00,2.565000e+04,6250.50,0.00,0.00,0.00
25%,34.00,1.125000e+05,66132.00,0.37,6.00,9.00
50%,43.00,1.530000e+05,135000.00,0.53,12.00,41.00
75%,53.00,2.025000e+05,315000.00,0.67,24.00,92.00
max,68.00,1.170000e+08,4050000.00,0.90,84.00,597.00


- **AGE_YEARS:** 20–68, mean 43, almost symmetric.
- **AMT_INCOME_TOTAL:** median 153K but max **117M** → extreme right skew (log transform in the model).
- **AMT_APPLICATION:** median 135K, mean 267K → right skewed (a few very large loans up to 4.05M).
- **EXT_SOURCE_3:** 0–0.90, median 0.53, **19% missing** (filled with the median in the model).
- **CNT_PAYMENT:** median 12 months, common terms are 6 / 12 / 24; **0 = revolving loans** (no fixed term).
- **MONTH_EMPLOYED:** median 41 months (~3.5 years); **0 = not working** (mostly pensioners).

## Business Questions

### Q1: What percentage of applications are refused?

In [78]:
# Aggregate first: plot 2 numbers, not 329K rows (keeps the notebook light)
status_counts = df_eda['NAME_CONTRACT_STATUS'].value_counts().reset_index()

px.pie(data_frame= status_counts, names= 'NAME_CONTRACT_STATUS', values= 'count',
       title= 'Approved vs Refused Applications')

- **Answer:** 84.6% of applications are approved and 15.4% refused → the target is **imbalanced**, so accuracy alone is misleading (Precision, Recall and F1 are used in the model).

In [79]:
def refusal_rate(col):
    """Number of applications and refusal rate (%) for each category of a column."""
    result = df_eda.groupby(col).agg(
        Applications = ('REFUSED', 'size'),
        Refusal_Rate = ('REFUSED', 'mean')
    ).reset_index()
    result['Refusal_Rate'] = (result['Refusal_Rate'] * 100).round(1)
    return result.sort_values('Refusal_Rate', ascending= False)

### Q2: Does the contract type affect refusal?

In [80]:
# Refusal rate (%) and number of applications per contract type
contract = refusal_rate('NAME_CONTRACT_TYPE')
contract

,NAME_CONTRACT_TYPE,Applications,Refusal_Rate
0,Cash loans,121265,27.2
2,Revolving loans,33779,26.8
1,Consumer loans,174119,4.9


In [81]:
px.bar(data_frame= contract, x= 'NAME_CONTRACT_TYPE', y= 'Refusal_Rate', text= 'Refusal_Rate',
       labels= {'NAME_CONTRACT_TYPE' : 'Contract Type', 'Refusal_Rate' : 'Refusal Rate (%)'},
       title= 'Refusal Rate by Contract Type')

- **Answer:** Yes, strongly. Consumer loans (buying goods) are refused only **4.9%** of the time because they are secured by the goods, while Cash (**27.2%**) and Revolving loans (**26.8%**) are refused ~5.5× more. This explains why `NAME_CONTRACT_TYPE` is the most important feature in the model.

### Q3: Are new clients refused less than repeat clients?

In [82]:
client = refusal_rate('NAME_CLIENT_TYPE')
client

,NAME_CLIENT_TYPE,Applications,Refusal_Rate
2,Repeater,220605,20.9
1,Refreshed,36215,8.1
0,New,71775,1.9


In [83]:
px.bar(data_frame= client, x= 'NAME_CLIENT_TYPE', y= 'Refusal_Rate', text= 'Refusal_Rate',
       labels= {'NAME_CLIENT_TYPE' : 'Client Type', 'Refusal_Rate' : 'Refusal Rate (%)'},
       title= 'Refusal Rate by Client Type')

- **Answer:** Yes. New clients are refused only **1.9%**, Refreshed **8.1%** and Repeaters **20.9%** (11× more than new clients). The bank seems to accept new clients more easily to win them, while repeat clients are judged on their history. This explains why `NAME_CLIENT_TYPE` is the second most important feature in the model.

### Q4: Does the requested amount differ between approved and refused applications?

In [84]:
# Statistics on all applications
df_eda.groupby('NAME_CONTRACT_STATUS')['AMT_APPLICATION'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
NAME_CONTRACT_STATUS,,,,,,,,
Approved,278563.0,231400.0,295457.0,6250.0,61020.0,119457.0,253942.0,4050000.0
Refused,50600.0,462085.0,455283.0,6916.0,135000.0,288000.0,675000.0,3825000.0


In [85]:
# Box plot on the 10K sample (log scale: amounts are highly skewed)
px.box(data_frame= df_sample, x= 'NAME_CONTRACT_STATUS', y= 'AMT_APPLICATION', color= 'NAME_CONTRACT_STATUS',
       log_y= True, labels= {'NAME_CONTRACT_STATUS' : 'Status', 'AMT_APPLICATION' : 'Requested Amount (log)'},
       title= 'Requested Amount: Approved vs Refused')

- **Answer:** Yes. Refused applications ask for much larger amounts: median **288K** vs **119K** for approved (2.4×), and the mean is 2× higher (462K vs 231K). Part of this comes from the contract type: cash loans are larger and refused more often than consumer loans.

### Q5: Does the credit score affect refusal?

In [86]:
# Mean credit scores on all applications
df_eda.groupby('NAME_CONTRACT_STATUS')[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean().round(3)

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
NAME_CONTRACT_STATUS,,,
Approved,0.502,0.515,0.522
Refused,0.497,0.503,0.440


In [87]:
# Histogram on the 10K sample: approved vs refused distributions
px.histogram(data_frame= df_sample, x= 'EXT_SOURCE_3', color= 'NAME_CONTRACT_STATUS',
             histnorm= 'percent', barmode= 'overlay', nbins= 40,
             labels= {'EXT_SOURCE_3' : 'Credit Score (EXT_SOURCE_3)', 'NAME_CONTRACT_STATUS' : 'Status'},
             title= 'Credit Score Distribution: Approved vs Refused')

- **Answer:** Only partly. `EXT_SOURCE_1` and `EXT_SOURCE_2` are almost the same for approved and refused (0.515 vs 0.503). Only **`EXT_SOURCE_3`** shows a clear gap (**0.522 vs 0.440**): refused clients have lower scores. This matches the model, where `EXT_SOURCE_3` is in the top 4 features.

### Q6: Does the refusal rate change with age?

In [88]:
# Age groups of 5 years, then refusal rate per group (aggregated → light plot)
df_eda['AGE_GROUP'] = pd.cut(df_eda['AGE_YEARS'], bins= range(20, 75, 5), right= False).astype(str)

age = refusal_rate('AGE_GROUP').sort_values('AGE_GROUP')
age

,AGE_GROUP,Applications,Refusal_Rate
0,"[20, 25)",13090,12.0
1,"[25, 30)",34901,14.6
2,"[30, 35)",42188,14.4
3,"[35, 40)",45896,14.6
4,"[40, 45)",44341,15.1
5,"[45, 50)",37931,15.9
6,"[50, 55)",37098,16.3
7,"[55, 60)",35148,17.5
8,"[60, 65)",29920,17.5
9,"[65, 70)",8650,12.3


In [89]:
px.line(data_frame= age, x= 'AGE_GROUP', y= 'Refusal_Rate', markers= True, text= 'Refusal_Rate',
        labels= {'AGE_GROUP' : 'Age Group', 'Refusal_Rate' : 'Refusal Rate (%)'},
        title= 'Refusal Rate by Age Group').update_traces(textposition= 'top center')

- **Answer:** The refusal rate slowly **increases with age**, from **12.0%** (20–25) to **17.5%** (55–65), then drops to **12.3%** after 65. The differences are small (~5 points), so age alone is a weak factor. This is the opposite of the usual "young = risky" idea, and may be linked to the type of loan older clients ask for (not verified).

### Q7: What is the relationship between income and the requested amount?

In [90]:
# Correlation on all applications (log: both columns are highly skewed)
np.log1p(df_eda[['AMT_INCOME_TOTAL', 'AMT_APPLICATION']]).corr().round(2)

,AMT_INCOME_TOTAL,AMT_APPLICATION
AMT_INCOME_TOTAL,1.00,0.24
AMT_APPLICATION,0.24,1.00


In [91]:
# Scatter on the 10K sample
px.scatter(data_frame= df_sample, x= 'AMT_INCOME_TOTAL', y= 'AMT_APPLICATION', color= 'NAME_CONTRACT_STATUS',
           log_x= True, log_y= True, opacity= 0.5,
           labels= {'AMT_INCOME_TOTAL' : 'Income (log)', 'AMT_APPLICATION' : 'Requested Amount (log)', 'NAME_CONTRACT_STATUS' : 'Status'},
           title= 'Income vs Requested Amount')

- **Answer:** Weak positive relationship (**correlation = 0.24** after log): higher income asks for slightly larger amounts, but many clients request amounts not related to their income.
- Refused applications (red) are concentrated at the **top** for every income level → a large amount **compared to income** is refused more often, which supports the new feature `LOAN_TO_INCOME`.
- The vertical lines show that incomes are recorded as **rounded values** (e.g. 112.5K, 135K, 225K).

### Q8: Do education and income type affect refusal?

In [92]:
edu = refusal_rate('NAME_EDUCATION_TYPE')
income_type = refusal_rate('NAME_INCOME_TYPE')

display(edu)
display(income_type)

,NAME_EDUCATION_TYPE,Applications,Refusal_Rate
3,Lower secondary,4045,16.0
4,Secondary / secondary special,236096,15.6
2,Incomplete higher,11123,15.2
1,Higher education,77713,14.6
0,Academic degree,186,9.7


,NAME_INCOME_TYPE,Applications,Refusal_Rate
5,Unemployed,16,62.5
2,Pensioner,60379,16.3
0,Commercial associate,73965,16.2
6,Working,171402,14.8
3,State servant,23385,14.6
1,Maternity leave,2,0.0
4,Student,14,0.0


In [93]:
px.bar(data_frame= edu, x= 'NAME_EDUCATION_TYPE', y= 'Refusal_Rate', text= 'Refusal_Rate',
       labels= {'NAME_EDUCATION_TYPE' : 'Education', 'Refusal_Rate' : 'Refusal Rate (%)'},
       title= 'Refusal Rate by Education')

In [94]:
px.bar(data_frame= income_type, x= 'NAME_INCOME_TYPE', y= 'Refusal_Rate', text= 'Refusal_Rate',
       labels= {'NAME_INCOME_TYPE' : 'Income Type', 'Refusal_Rate' : 'Refusal Rate (%)'},
       title= 'Refusal Rate by Income Type')

- **Education:** refusal slightly decreases with education level, from **16.0%** (lower secondary) to **14.6%** (higher education). Academic degree is lower (9.7%) but has only 186 applications.
- **Income type:** the main groups are close (**14.6%–16.3%**). Unemployed shows 62.5% but with only **16 applications**, so it is not reliable.
- **Conclusion:** education and income type have a weak effect on refusal. Rates based on very small groups should not be trusted.

### Q9: How are the numerical columns related to each other and to refusal?

In [95]:
corr_cols = ['AMT_INCOME_TOTAL', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_ANNUITY', 'CNT_PAYMENT',
             'AGE_YEARS', 'MONTH_EMPLOYED', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
             'EXT_SOURCE_2', 'EXT_SOURCE_3', 'REFUSED']

corr = df_eda[corr_cols].corr().round(2)

px.imshow(corr, text_auto= True, color_continuous_scale= 'RdBu_r', zmin= -1, zmax= 1,
          title= 'Correlation Heatmap', width= 900, height= 800)

- **Strongest links with refusal:** `CNT_PAYMENT` (**0.26**), `AMT_APPLICATION` (**0.25**) → longer and larger loans are refused more. `EXT_SOURCE_3` (**−0.15**) → higher score, fewer refusals. Income, age and `EXT_SOURCE_2` are almost **0**.
- **Multicollinearity:** `AMT_APPLICATION` – `AMT_CREDIT` = **0.99** and – `AMT_ANNUITY` = **0.82** (both dropped from the model as leakage). `CNT_CHILDREN` – `CNT_FAM_MEMBERS` = **0.88** (almost the same information).
- `AGE_YEARS` – `CNT_CHILDREN` = **−0.34**: older clients have fewer children at home.

## 📊 EDA Summary
- **Target:** 15.4% refused → imbalanced (Precision / Recall / F1 used in the model).
- **Strongest factors (the application):** contract type (consumer 4.9% vs cash 27.2%), client type (new 1.9% vs repeater 20.9%), requested amount (refused median 2.4× higher) and loan term (`CNT_PAYMENT`, corr 0.26).
- **Weaker factors (the client):** `EXT_SOURCE_3` (0.52 vs 0.44), age (12% → 17.5%), education and income type (14.6%–16.3%).
- **Key insight:** refusal depends more on **what is requested** than on **who requests it**. This matches the model's feature importance and explains why client segments have similar refusal rates.